## Bridge

---

> **In one line.** A Bridge splits one tangled hierarchy into two independent sets — the **abstractions** $\mathcal{A}$ and the **implementations** $\mathcal{B}$ — and joins them by *composition* rather than inheritance: each abstraction $A_i$ holds a reference to some implementation $B_j$. The combinatorial cost collapses from $m \times n$ classes to $m + n$.

### 1. The two independent sets

Let $\mathcal{A}$ be the set of **abstractions** — the high-level concepts a client manipulates, e.g. $\mathcal{A} = \{\text{Circle}, \text{Square}\}$ — and let $\mathcal{B}$ be the set of **implementations** — the platform-specific engines that do the real work, e.g. $\mathcal{B} = \{\text{SVGRenderer}, \text{CanvasRenderer}\}$. Write their sizes as

$$|\mathcal{A}| = m, \qquad |\mathcal{B}| = n.$$

A working configuration is a pair $A_i \times B_j$ — one abstraction running on one implementation, such as a Circle drawn by an SVGRenderer. The space of all possible behaviours the system must deliver is the Cartesian product

$$\mathcal{A} \times \mathcal{B} \;=\; \{\, A_i \times B_j : A_i \in \mathcal{A},\; B_j \in \mathcal{B} \,\}, \qquad |\mathcal{A} \times \mathcal{B}| = m \cdot n.$$

### 2. Why inheritance explodes

If every combination is modelled as its own class — `CircleSVG`, `CircleCanvas`, `SquareSVG`, … — then realizing the full product requires **one class per cell**, i.e. $m \times n$ classes. The Bridge instead realizes the *same* $m \times n$ behaviours by composition: it keeps the $m$ abstractions and the $n$ implementations as separate hierarchies and lets each $A_i$ **hold a reference** to a $B_j$. The decisive count is therefore

$$\boxed{\;\text{without Bridge: } m \times n \text{ classes} \qquad\Longrightarrow\qquad \text{with Bridge: } m + n \text{ classes}\;}$$

and crucially every cell of the product is still reachable:

$$A_i \text{ holds a reference to } B_j \quad\Longrightarrow\quad \text{all } m \times n \text{ combinations achievable.}$$

The reference itself is the *bridge*. Data flows from the high-level call down across that reference into the platform:

$$\text{client} \xrightarrow{\;A_i\;} \underbrace{A_i.\text{op}}_{\text{abstraction}} \xrightarrow{\;\text{delegates}\;} \underbrace{B_j.\text{op}}_{\text{implementation}}.$$

### 3. Composition decouples the dimensions

Because $\mathcal{A}$ and $\mathcal{B}$ are stored separately and linked only by a runtime reference, the two dimensions vary **independently**: an abstraction never names a concrete implementation, and an implementation never names an abstraction. The chosen $B_j$ can even be swapped on a live $A_i$ at runtime.

1. **Independent variation** — adding a new abstraction $A_{m+1}$ costs exactly $1$ new class; adding a new implementation $B_{n+1}$ also costs exactly $1$ new class. Neither change touches the other side.
   $$m \mapsto m+1 \;\Rightarrow\; +1 \text{ class}, \qquad n \mapsto n+1 \;\Rightarrow\; +1 \text{ class}.$$
2. **Composition over inheritance** — each $A_i$ holds the reference (the "bridge") to some $B_j$, and this reference can be reassigned at runtime; the link is *has-a*, never *is-a*.
3. **Savings grow quadratically** — the gap $m \cdot n - (m + n)$ widens fast. At $m = 4,\, n = 4$ inheritance needs $16$ classes while Bridge needs $8$; at $m = 10,\, n = 10$ it is $100$ versus $20$.

&nbsp;

> 🎨 Shapes on surfaces. $\mathcal{A} = \{\text{Circle, Square}\}$, $\mathcal{B} = \{\text{PDF, SVG, Screen}\}$. Without Bridge: $2 \times 3 = 6$ classes. With Bridge: $2 + 3 = 5$ classes. Add Triangle → $1$ new class, not $3$.

### Exercise 11 — Shape Renderer Bridge

---

**Scenario:** Draw Circle and Square ($\mathcal{A}$) on SVGRenderer and CanvasRenderer ($\mathcal{B}$). Goal: $m + n = 4$ classes achieving $m \times n = 4$ combinations.

**Your task:** Implement the Bridge so shapes and renderers are fully decoupled. Adding a Triangle should require exactly 1 new class.

```python
circle = Circle(radius=5, renderer=SVGRenderer())
circle.draw()   # SVG: drawing circle r=5
```

**Hints**

- The abstraction ($A_i$) holds `self.renderer = renderer` — this is the reference to $B_j$, the "bridge". The shape calls `self.renderer.render_circle()` to delegate platform work.
- Add Triangle and count: only 1 new class — not 2. This confirms the $m + n$ condition.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Implementations (set B) — the platform-specific engines. You do not change these.

class Renderer(ABC):                             # interface for every B_j
    @abstractmethod
    def render_circle(self, radius): ...
    @abstractmethod
    def render_square(self, side): ...

class SVGRenderer(Renderer):                     # B_1
    def render_circle(self, radius):
        print(f"SVG: drawing circle r={radius}")
    def render_square(self, side):
        print(f"SVG: drawing square s={side}")

class CanvasRenderer(Renderer):                  # B_2
    def render_circle(self, radius):
        print(f"Canvas: drawing circle r={radius}")
    def render_square(self, side):
        print(f"Canvas: drawing square s={side}")

# --------------------------------
# Abstractions (set A) — each A_i holds a reference (the bridge) to some B_j.

class Shape(ABC):
    def __init__(self, renderer):
        self.renderer = renderer                 # the bridge: reference to B_j
    @abstractmethod
    def draw(self): ...

class Circle(Shape):                             # A_1
    def __init__(self, radius, renderer):
        super().__init__(renderer)
        self.radius = radius
    def draw(self):
        # delegate platform work to B_j across the bridge
        ...                                      # e.g. self.renderer.render_circle(self.radius)

class Square(Shape):                             # A_2
    def __init__(self, side, renderer):
        super().__init__(renderer)
        self.side = side
    def draw(self):
        ...                                      # delegate: self.renderer.render_square(self.side)

# --------------------------------
circle = Circle(radius=5, renderer=SVGRenderer())
circle.draw()                                    # expect: SVG: drawing circle r=5

square = Square(side=3, renderer=CanvasRenderer())
square.draw()                                    # expect: Canvas: drawing square s=3

### Exercise 12 — Notification Bridge

---

**Scenario:** $\mathcal{A} = \{\text{AlertNotification, ReminderNotification}\}$, $\mathcal{B} = \{\text{EmailSender, SMSSender}\}$. Any notification type must work with any sender.

**Your task:** Apply Bridge so new notification types and new senders can be added independently.

```python
alert = AlertNotification(EmailSender())
alert.notify("Server down")   # any A_i composed with any B_j
```

**Hints**

- Count: $m = 2$ notification types, $n = 2$ senders. Bridge: $2 + 2 = 4$ classes. Adding a `PushNotificationSender` costs 1 class, not $m = 2$.
- Each notification ($A_i$) holds `self.sender = sender` (the bridge to $B_j$) and delegates the actual delivery via `self.sender.send(...)`.

In [ ]:
from abc import ABC, abstractmethod

# --------------------------------
# Implementations (set B) — the senders. You do not change these.

class Sender(ABC):                               # interface for every B_j
    @abstractmethod
    def send(self, message): ...

class EmailSender(Sender):                       # B_1
    def send(self, message):
        print(f"Email: {message}")

class SMSSender(Sender):                         # B_2
    def send(self, message):
        print(f"SMS: {message}")

# --------------------------------
# Abstractions (set A) — each A_i bridges to some B_j via self.sender.

class Notification(ABC):
    def __init__(self, sender):
        self.sender = sender                     # the bridge: reference to B_j
    @abstractmethod
    def notify(self, text): ...

class AlertNotification(Notification):           # A_1
    def notify(self, text):
        # build the alert message, then delegate delivery to B_j
        ...                                      # e.g. self.sender.send(f"[ALERT] {text}")

class ReminderNotification(Notification):        # A_2
    def notify(self, text):
        ...                                      # delegate: self.sender.send(f"[Reminder] {text}")

# --------------------------------
AlertNotification(EmailSender()).notify("Server down")
AlertNotification(SMSSender()).notify("Server down")
ReminderNotification(EmailSender()).notify("Standup at 10am")